# Arquitectura de Datos como Habilitador de Cultura Data-Driven

## Objetivos
- Comprender cómo decisiones arquitectónicas facilitan (u obstaculizan) adopción de datos.
- Diseñar plataformas self-service que democratizan acceso sin comprometer seguridad.
- Implementar catálogos de datos para descubrimiento autónomo.
- Usar contratos de datos para generar confianza entre productores y consumidores.
- Medir adopción de datos con métricas objetivas (engagement, self-service ratio).

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Diagramas de Componentes y Flujos de Datos (Aplicado)](12_diagramas_componentes_flujos_datos.ipynb) | Siguiente: [Observabilidad y Evolución de Arquitectura de Datos (Aplicado)](14_observabilidad_evolucion_arquitectura.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. Objetivos
2. 1. Arquitectura de Datos y Cultura: La Conexión
3. 2. Self-Service: Democratización sin Caos
4. 3. Catálogo de Datos: Google para tus Datos
5. 4. Código: Simulador de Catálogo con Búsqueda
6. 5. Contratos de Datos: SLAs para Confianza
7. 6. Observabilidad para Confianza
8. 7. Código: Métricas de Adopción

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Arquitectura de Datos como Habilitador de Cultura Data-Driven"] --> C["Notebook siguiente"]
```


## 1. Arquitectura de Datos y Cultura: La Conexión

### Problema Común
**"Queremos ser data-driven, pero..."**
- Los analistas esperan 2 semanas para acceso a datos.
- Nadie sabe qué datasets existen o qué significan las columnas.
- Los dashboards muestran números contradictorios entre equipos.
- Solo el equipo de TI puede crear nuevos pipelines.

**Diagnóstico:** La arquitectura no está diseñada para democratización.

### Solución: Principios Arquitectónicos
| Principio                  | Implementación                                      |
|----------------------------|----------------------------------------------------|
| **Self-Service**           | Catálogo + queries pre-aprobadas + sandboxes       |
| **Descubrimiento**         | Metadata management + búsqueda semántica           |
| **Confianza**              | Data quality + lineage + SLAs documentados         |
| **Seguridad por diseño**   | RBAC + data masking + auditoría automática         |
| **Observabilidad**         | Dashboards de uso + métricas de calidad            |

## 2. Self-Service: Democratización sin Caos

### Definición
**Self-service** = usuarios no técnicos pueden acceder, explorar y analizar datos sin depender de TI.

### Componentes
1. **Catálogo de datos:** Directorio searchable de todos los datasets.
2. **Query builder / BI tool:** Interfaz visual para construir consultas (sin SQL).
3. **Sandbox environments:** Espacios seguros para experimentar sin afectar producción.
4. **Pre-built queries:** Consultas certificadas para casos de uso comunes.

### Arquitectura Self-Service
```mermaid
graph TD
    User[Usuario Negocio] --> Catalog[Catálogo de Datos]
    Catalog --> SearchAPI[API Búsqueda]
    SearchAPI --> Metadata[(Metadata Store)]
    
    User --> BI[Herramienta BI]
    BI --> Semantic[Capa Semántica]
    Semantic --> Gold[(Gold Layer - Curated)]
    
    User --> Sandbox[Sandbox Jupyter/SQL]
    Sandbox --> Silver[(Silver Layer - Validated)]
    
    RBAC[Control de Acceso] -.->|Autoriza| Catalog
    RBAC -.->|Autoriza| BI
    RBAC -.->|Autoriza| Sandbox
    
    style Gold fill:#9f9
    style Silver fill:#bbf
    style RBAC fill:#f99
```

### Niveles de Self-Service
| Nivel       | Capacidad                            | Herramienta Ejemplo       |
|------------|--------------------------------------|---------------------------|
| **Básico**  | Dashboards pre-construidos          | Tableau, Power BI         |
| **Intermedio** | Exploración con filtros/drill-down | Looker, Metabase          |
| **Avanzado** | Construir queries SQL propias        | dbt Cloud, Redash         |
| **Experto**  | Notebooks para análisis custom       | Jupyter, Databricks       |

## 3. Catálogo de Datos: Google para tus Datos

### Metadatos Clave
- **Técnicos:** Esquema (columnas, tipos), ubicación (S3 bucket, tabla Redshift).
- **De negocio:** Descripción, owner, dominio, casos de uso.
- **Operacionales:** Frescura (última actualización), calidad (% nulos), uso (queries/día).

### Funcionalidades Críticas
1. **Búsqueda semántica:** "clientes activos últimos 30 días" → sugiere tabla `gold.customer_activity`.
2. **Lineage visual:** Desde dashboard → tabla → pipeline → fuente.
3. **Colaboración:** Comentarios, ratings, FAQs.
4. **Alertas:** Notificar cuando dataset cambia esquema o cae calidad.

### Herramientas
- **Open Source:** Apache Atlas, Amundsen, DataHub, OpenMetadata.
- **Comercial:** Alation, Collibra, Atlan, Informatica.
- **Cloud-native:** AWS Glue Catalog, Azure Purview, Google Dataplex.

## 4. Código: Simulador de Catálogo con Búsqueda

In [ ]:
from dataclasses import dataclass
from typing import List, Optional
from datetime import datetime, timedelta
import random

@dataclass
class Dataset:
    id: str
    name: str
    description: str
    domain: str  # Marketing, Sales, Finance, etc.
    owner: str
    layer: str  # Bronze, Silver, Gold
    last_updated: datetime
    quality_score: float  # 0-100
    monthly_queries: int
    tags: List[str]
    
class DataCatalog:
    def __init__(self):
        self.datasets: List[Dataset] = []
    
    def register(self, dataset: Dataset):
        self.datasets.append(dataset)
    
    def search(self, query: str, domain: Optional[str] = None) -> List[Dataset]:
        """Búsqueda simple por keywords en nombre/descripción/tags."""
        results = []
        query_lower = query.lower()
        
        for ds in self.datasets:
            # Filtro por dominio si se especifica
            if domain and ds.domain != domain:
                continue
            
            # Búsqueda en nombre, descripción y tags
            if (query_lower in ds.name.lower() or 
                query_lower in ds.description.lower() or
                any(query_lower in tag.lower() for tag in ds.tags)):
                results.append(ds)
        
        # Ordenar por popularidad (queries) y calidad
        return sorted(results, key=lambda x: (x.monthly_queries * x.quality_score), reverse=True)
    
    def get_most_used(self, top_n: int = 5) -> List[Dataset]:
        return sorted(self.datasets, key=lambda x: x.monthly_queries, reverse=True)[:top_n]
    
    def get_stale_datasets(self, days: int = 7) -> List[Dataset]:
        """Datasets no actualizados en N días."""
        threshold = datetime.now() - timedelta(days=days)
        return [ds for ds in self.datasets if ds.last_updated < threshold]

# Ejemplo: Poblar catálogo
catalog = DataCatalog()

catalog.register(Dataset(
    id="gold.customer_360",
    name="Customer 360",
    description="Vista unificada de clientes con datos de CRM, transacciones y soporte",
    domain="Sales",
    owner="sales-analytics@company.com",
    layer="Gold",
    last_updated=datetime.now() - timedelta(hours=2),
    quality_score=95,
    monthly_queries=1500,
    tags=["customer", "crm", "360", "high-quality"]
))

catalog.register(Dataset(
    id="gold.monthly_revenue",
    name="Monthly Revenue",
    description="Ingresos mensuales agregados por región y producto",
    domain="Finance",
    owner="finance-team@company.com",
    layer="Gold",
    last_updated=datetime.now() - timedelta(days=1),
    quality_score=98,
    monthly_queries=800,
    tags=["revenue", "finance", "kpi", "executive"]
))

catalog.register(Dataset(
    id="silver.web_events",
    name="Web Events (Cleaned)",
    description="Eventos de navegación web validados y enriquecidos con geo-IP",
    domain="Marketing",
    owner="marketing-ops@company.com",
    layer="Silver",
    last_updated=datetime.now() - timedelta(hours=1),
    quality_score=85,
    monthly_queries=2500,
    tags=["web", "events", "marketing", "real-time"]
))

# Búsqueda
print("=== Búsqueda: 'customer' ===")
results = catalog.search("customer")
for ds in results:
    print(f"- {ds.name} ({ds.layer}) | Owner: {ds.owner} | Quality: {ds.quality_score}%")

print("\n=== Datasets Más Usados ===")
for ds in catalog.get_most_used(3):
    print(f"- {ds.name}: {ds.monthly_queries} queries/mes")

print("\n=== Datasets Desactualizados (>7 días) ===")
stale = catalog.get_stale_datasets(7)
if stale:
    for ds in stale:
        print(f"- {ds.name}: última actualización hace {(datetime.now() - ds.last_updated).days} días")
else:
    print("✓ Todos los datasets están actualizados")

## 5. Contratos de Datos: SLAs para Confianza

### Problema
**Consumidor (Analista):** "El dashboard muestra ceros esta mañana, ¿qué pasó?"
**Productor (Ingeniero):** "Cambié el schema, ahora `revenue` se llama `total_revenue`".

**Impacto:** Pérdida de confianza → menos adopción.

### Solución: Data Contracts
Acuerdo explícito entre productor y consumidor:
- **Schema:** Columnas, tipos, constraints.
- **SLA:** Frescura (actualización cada 1h), disponibilidad (99.5%).
- **Quality:** % completitud, rangos válidos.
- **Breaking changes:** Notificación 2 semanas antes, versionado.

### Ejemplo de Contrato
```yaml
dataset: gold.daily_sales
version: 2.1.0
owner: sales-analytics@company.com
sla:
  freshness: "Updated daily at 08:00 UTC"
  availability: 99.5%
schema:
  - name: date
    type: DATE
    nullable: false
  - name: region
    type: STRING
    nullable: false
  - name: total_revenue
    type: DECIMAL(12,2)
    nullable: false
    constraint: "> 0"
quality:
  completeness: ">= 99%"
  uniqueness: "date + region unique"
consumers:
  - executive-dashboard
  - finance-reports
change_policy:
  breaking_changes_notice: "14 days"
  deprecation_period: "90 days"
```

## 6. Observabilidad para Confianza

### Métricas de Adopción
1. **Engagement:**
   - % empleados con acceso al catálogo.
   - Queries ejecutadas/semana.
   - Usuarios activos en BI tool.

2. **Self-Service Ratio:**
   - % de consultas resueltas sin ticket a TI.
   - Tiempo promedio de solicitud de acceso (debe bajar).

3. **Data Quality Perception:**
   - Encuesta trimestral: "¿Confías en los datos?" (1-5).
   - Incidentes de calidad reportados/mes (debe bajar).

4. **Time-to-Insight:**
   - Días desde pregunta de negocio hasta dashboard.
   - Benchmark: <3 días para análisis exploratorio, <1 día para query ad-hoc.

### Dashboard de Adopción
```
╔════════════════════════════════════════════════════╗
║      Data Platform Adoption Dashboard             ║
╠════════════════════════════════════════════════════╣
║ Active Users (Last 30d):  450 / 600  (75%) ↑      ║
║ Self-Service Ratio:       82%          ↑          ║
║ Avg Time-to-Insight:      2.3 days     ↓          ║
║ Data Trust Score:         4.2 / 5.0    ↑          ║
║                                                    ║
║ Top Datasets (by queries):                         ║
║   1. Customer 360        1,500 queries/mo         ║
║   2. Web Events          2,500 queries/mo         ║
║   3. Monthly Revenue       800 queries/mo         ║
║                                                    ║
║ Quality Alerts (Last 7d): 2 (down from 8)        ║
╚════════════════════════════════════════════════════╝
```

## 7. Código: Métricas de Adopción

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
import random

@dataclass
class AdoptionMetrics:
    total_employees: int
    active_users_30d: int
    total_queries_month: int
    queries_self_service: int  # Sin ticket a TI
    queries_via_ticket: int
    avg_time_to_insight_days: float
    data_trust_score: float  # 1-5 scale
    quality_incidents_month: int
    
    def engagement_rate(self) -> float:
        return (self.active_users_30d / self.total_employees) * 100
    
    def self_service_ratio(self) -> float:
        total = self.queries_self_service + self.queries_via_ticket
        return (self.queries_self_service / total * 100) if total > 0 else 0
    
    def evaluate(self) -> dict:
        """Evaluar madurez de adopción."""
        score = 0
        feedback = []
        
        # Engagement
        engagement = self.engagement_rate()
        if engagement >= 70:
            score += 25
            feedback.append(f"✓ Engagement alto ({engagement:.1f}%)")
        elif engagement >= 50:
            score += 15
            feedback.append(f"⚠ Engagement moderado ({engagement:.1f}%), objetivo: 70%")
        else:
            feedback.append(f"✗ Engagement bajo ({engagement:.1f}%), objetivo: 70%")
        
        # Self-Service
        ss_ratio = self.self_service_ratio()
        if ss_ratio >= 80:
            score += 25
            feedback.append(f"✓ Self-service alto ({ss_ratio:.1f}%)")
        elif ss_ratio >= 60:
            score += 15
            feedback.append(f"⚠ Self-service moderado ({ss_ratio:.1f}%), objetivo: 80%")
        else:
            feedback.append(f"✗ Self-service bajo ({ss_ratio:.1f}%), objetivo: 80%")
        
        # Time-to-Insight
        if self.avg_time_to_insight_days <= 3:
            score += 25
            feedback.append(f"✓ Time-to-insight rápido ({self.avg_time_to_insight_days} días)")
        elif self.avg_time_to_insight_days <= 5:
            score += 15
            feedback.append(f"⚠ Time-to-insight aceptable ({self.avg_time_to_insight_days} días), objetivo: ≤3")
        else:
            feedback.append(f"✗ Time-to-insight lento ({self.avg_time_to_insight_days} días), objetivo: ≤3")
        
        # Confianza
        if self.data_trust_score >= 4.0:
            score += 25
            feedback.append(f"✓ Confianza alta ({self.data_trust_score}/5.0)")
        elif self.data_trust_score >= 3.0:
            score += 15
            feedback.append(f"⚠ Confianza moderada ({self.data_trust_score}/5.0), objetivo: ≥4.0")
        else:
            feedback.append(f"✗ Confianza baja ({self.data_trust_score}/5.0), objetivo: ≥4.0")
        
        # Nivel de madurez
        if score >= 90:
            maturity = "Optimizado (world-class)"
        elif score >= 70:
            maturity = "Avanzado (data-driven)"
        elif score >= 50:
            maturity = "Intermedio (en progreso)"
        else:
            maturity = "Inicial (cultura tradicional)"
        
        return {
            "score": score,
            "maturity": maturity,
            "feedback": feedback
        }

# Ejemplo: Empresa en transformación (6 meses)
metrics_current = AdoptionMetrics(
    total_employees=600,
    active_users_30d=450,
    total_queries_month=5000,
    queries_self_service=4100,
    queries_via_ticket=900,
    avg_time_to_insight_days=2.3,
    data_trust_score=4.2,
    quality_incidents_month=2
)

result = metrics_current.evaluate()
print("=== Evaluación de Adopción Data-Driven ===")
print(f"Score: {result['score']}/100")
print(f"Madurez: {result['maturity']}")
print("\nDetalles:")
for fb in result['feedback']:
    print(f"  {fb}")

print("\n=== Comparación vs Baseline (inicio) ===")
print("Métrica                | Inicio  | Actual  | Mejora")
print("----------------------|---------|---------|--------")
print(f"Engagement            | 30%     | {metrics_current.engagement_rate():.1f}%   | +{metrics_current.engagement_rate()-30:.1f}%")
print(f"Self-service ratio    | 20%     | {metrics_current.self_service_ratio():.1f}%   | +{metrics_current.self_service_ratio()-20:.1f}%")
print(f"Time-to-insight       | 8d      | {metrics_current.avg_time_to_insight_days}d     | -5.7d")
print(f"Data trust score      | 2.5/5   | {metrics_current.data_trust_score}/5   | +1.7")

## 8. Patrones Anti-Pattern a Evitar

| Anti-Pattern              | Impacto                                  | Solución                          |
|---------------------------|------------------------------------------|-----------------------------------|
| **Data Swamp**            | Usuarios no encuentran datos confiables  | Catálogo + governance + calidad   |
| **Shadow IT**             | Equipos crean BDs propias (silos)       | Plataforma única, incentivos      |
| **Acceso todo o nada**    | O no tienen acceso, o acceso a todo     | RBAC granular, data masking       |
| **Documentation drift**   | Docs desactualizados (nadie los lee)    | Metadata auto-generado, lineage   |
| **Single point of failure** | Solo 1 persona sabe cómo funciona     | Democratización, ownership distribuido |

## 9. Ejercicio Integrador
**Contexto:** Tu empresa tiene 500 empleados, 15% usa datos regularmente, 80% de consultas requieren ticket a TI (tiempo promedio: 5 días).

**Tareas:**
1. Diseña arquitectura self-service con 3 capas (Bronze/Silver/Gold) indicando qué roles acceden a cada una.
2. Define 5 metadatos clave para el catálogo de datos.
3. Propón un contrato de datos para dataset `gold.customer_lifetime_value` (schema + SLA + quality).
4. Define 3 métricas para medir éxito en 6 meses.
5. Estima ROI: si self-service reduce tickets 60% y cada ticket cuesta $50 (tiempo analista), ¿cuál es el ahorro anual?

## 10. Resumen
- **Arquitectura habilita cultura:** Self-service, catálogo, contratos, observabilidad.
- **Self-service democratiza acceso** sin comprometer seguridad (RBAC + sandboxes).
- **Catálogo de datos** = Google para datasets (búsqueda, lineage, colaboración).
- **Contratos de datos** generan confianza entre productores y consumidores.
- **Métricas de adopción:** Engagement, self-service ratio, time-to-insight, trust score.

**Siguiente (Senior):** Framework de transformación organizacional, change management, ROI estratégico.

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- Referencias del curso: ../04-recursos/referencias.md
